# 📖 Grok 长篇小说自动写作工具

将你的角色设定和故事大纲，自动扩展为 10 章、每章约 8000 字的长篇小说，并导出为 PDF。

**使用前请先完成：**
1. 安装依赖：`pip install -r requirements.txt`
2. 在下方填入你的 Grok API Key
3. 编辑 `user_inputs/` 目录下的角色设定、情节、风格文件
4. 编写 `chapters/` 目录下的章节大纲文件（每章一个 .txt）

---
## 步骤 0：配置 Grok API

In [ ]:
# ============================================================
# ✏️ 请在此填入你的 Grok API Key
# ============================================================
# 从 x.ai 获取: https://console.x.ai

GROK_API_KEY = "xai-..."  # <-- 替换为你的真实 Key

# 可选：修改 API 参数
GROK_BASE_URL = "https://api.x.ai/v1"
GROK_MODEL = "grok-3-mini"       # 或 grok-3, grok-2-latest
GROK_TEMPERATURE = 1.0
GROK_MAX_TOKENS = 16384

print(f"✅ API 配置就绪，模型: {GROK_MODEL}")

---
## 步骤 0.5：快速测试 Grok API（一问一答）

用 `client.chat()` 测试 API 连通性：

In [ ]:
from grok_client import GrokClient
import config as cfg

cfg.API_BASE_URL = GROK_BASE_URL
cfg.API_MODEL = GROK_MODEL
cfg.API_TEMPERATURE = GROK_TEMPERATURE
cfg.API_MAX_TOKENS = GROK_MAX_TOKENS

client = GrokClient(
    api_key=GROK_API_KEY,
    base_url=cfg.API_BASE_URL,
    model=cfg.API_MODEL,
    temperature=cfg.API_TEMPERATURE,
    max_tokens=cfg.API_MAX_TOKENS,
    timeout=cfg.API_TIMEOUT,
    max_retries=3,
)

# 简单测试：一问一答
reply = client.chat("用一句话描述写小说的乐趣")
print(reply)

---
## 步骤 1：输入角色设定与故事内容

编辑 `user_inputs/` 目录下的三个文件后，运行下方单元格加载。

In [ ]:
def load_txt(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

CHARACTERS = load_txt('user_inputs/characters.txt')
PLOT = load_txt('user_inputs/plot.txt')
STYLE = load_txt('user_inputs/style.txt')

print(f"✅ 角色设定: {len(CHARACTERS)} 字")
print(f"✅ 故事大纲: {len(PLOT)} 字")
print(f"✅ 写作风格: {len(STYLE)} 字")

---
## 步骤 2：初始化小说生成器

In [ ]:
from story_generator import NovelGenerator
from pdf_exporter import export_novel_to_pdf

generator = NovelGenerator(
    grok_client=client,
    characters=CHARACTERS,
    plot=PLOT,
    style=STYLE,
    output_dir="outputs",
    save_intermediates=True,
)

print("✅ 初始化完成")

---
## 步骤 3：加载章节大纲

从 `chapters/` 目录读取你自行编写的章节大纲文件。
每个文件格式：第一行为标题（如 `# 第1章 启程`），后续内容为本章大纲。

In [ ]:
chapter_outlines = generator.load_chapters_from_dir()

print(f"\n{'=' * 50}")
print(f"📖 已加载 {len(chapter_outlines)} 章大纲")
print(f"{'=' * 50}")
for ch in chapter_outlines:
    print(f"\n  📄 第 {ch['chapter']} 章：{ch['title']}")
    preview = ch['outline'][:120] + "..." if len(ch['outline']) > 120 else ch['outline']
    print(f"     {preview}")

---
## 步骤 4：每章拆分为 10 小段

对每一章的大纲，调用 Grok API 拆分为 10 小段的情节要点。

In [ ]:
generator.generate_all_segments()

---
## 步骤 5：逐段写作全部章节

针对每个小段的情节要点，逐段生成正文。
**⚠️ 耗时最长（10 章 × 10 段 ≈ 100 次 API 调用）。**

In [ ]:
generator.write_all_chapters()

total_chars = len(generator.full_novel)
print(f"\n{'=' * 50}")
print(f"🎉 小说全文完成！")
print(f"   总字数：约 {total_chars} 字")
print(f"{'=' * 50}")

---
## 步骤 6：导出 PDF

In [ ]:
pdf_path = export_novel_to_pdf(
    novel_text=generator.full_novel,
    chapter_outlines=chapter_outlines,
    output_path="outputs/novel.pdf",
    title="长篇小说",
    author="Grok 自动写作",
)

print(f"\n📄 PDF 已保存至: {pdf_path}")
print("✅ 全部流程完成！")

---
## 附录 A：直接用 Grok 聊天（一问一答）

如果你只是想和 Grok 聊两句，不需要完整的小说管线，用这个：

In [ ]:
reply = client.chat("你的问题写在这里")
print(reply)

---
## 附录 B：断点续写

In [ ]:
# 如果写作中断，修改 start_chapter 从指定章继续
# start_chapter = 3  # 从第 4 章开始（索引从 0）
# for idx in range(start_chapter, len(generator.chapter_outlines)):
#     generator.write_chapter(idx)
# generator.assemble_novel()
# print("✅ 续写完成")